# OneVoice V2 — MT error analysis
Đọc `predictions.csv` đã có trên Drive, thống kê lỗi critical field và đưa ví dụ để sửa Context/Safety hoặc quyết định fine-tune. Đặt `MODEL_LABEL` và `DIRECTION` cho candidate cần xem. Notebook này không tải model và không chạy lại benchmark.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO)
REPORT_ROOT = DRIVE_ROOT / 'reports/mt'
# Change both values together to inspect another completed candidate.
MODEL_LABEL = 'candidate_en2vi'
DIRECTION = 'en2vi'
OUTPUT = REPORT_ROOT / f'error_analysis/{MODEL_LABEL}_{DIRECTION}.json'


In [ ]:
INPUTS = [
    REPORT_ROOT / MODEL_LABEL / DIRECTION / suite / route / 'predictions.csv'
    for suite in ('test', 'minimal', 'safety')
    for route in ('raw', 'context')
]
missing = [str(path) for path in INPUTS if not path.is_file()]
if missing:
    raise FileNotFoundError('Run colab_mt_v2.ipynb first. Missing: ' + ', '.join(missing))
subprocess.run([sys.executable, 'scripts/analyze_mt_errors.py', '--inputs', *map(str, INPUTS), '--output', str(OUTPUT), '--top', '30'], check=True)
analysis = json.loads(OUTPUT.read_text(encoding='utf-8'))
analysis
